In [4]:
!pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com

Looking in indexes: https://pypi.org/simple, https://pypi.nvidia.com


In [7]:
import pandas as pd
import numpy as np
import time

n = 5000000
categories = ["Water Supply", "Sewage and Drainage", "Road and Infrastructure", "Electricity", "Sanitation and Garbage", "Corruption"]
data = {
    "id": range(n),
    "category": np.random.choice(categories, n),
    "status": np.random.choice(["Pending", "Resolved"], n),
    "priority": np.random.randint(1, 6, n)
}
df_pandas = pd.DataFrame(data)
df_pandas.to_csv("grievances_sample.csv", index=False)
print(f"Generated {n} sample records")

Generated 5000000 sample records


In [9]:
# Heavier operation: filter + sort + multiple aggregations
start = time.time()
filtered_cpu = df_pandas[df_pandas["status"] == "Pending"]
agg_cpu = filtered_cpu.groupby("category").agg({"priority": ["mean", "max", "count"]})
agg_cpu_sorted = agg_cpu.sort_values(("priority", "count"), ascending=False)
cpu_time2 = time.time() - start
print(f"Pandas (CPU) heavier op time: {cpu_time2:.4f} seconds")

start = time.time()
filtered_gpu = df_gpu[df_gpu["status"] == "Pending"]
agg_gpu = filtered_gpu.groupby("category").agg({"priority": ["mean", "max", "count"]})
gpu_time2 = time.time() - start
print(f"cuDF (GPU) heavier op time: {gpu_time2:.4f} seconds")

print(f"\nSpeedup on heavier operation: {cpu_time2/gpu_time2:.1f}x faster with RAPIDS")


Pandas (CPU) heavier op time: 0.8177 seconds
cuDF (GPU) heavier op time: 0.0854 seconds

Speedup on heavier operation: 9.6x faster with RAPIDS
